In [ ]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')

# ==============================
# 2. Load Dataset
# ==============================
df = pd.read_csv('Twitter_Data.csv')  
# Remove rows where category is missing
df = df.dropna(subset=['category'])

# Reset index (clean)
df = df.reset_index(drop=True)

print("Dataset Shape:", df.shape)
print(df.head())

# ==============================
# 3. Data Understanding
# ==============================
print("\nColumns:", df.columns)

print("\nClass Distribution:")
print(df['category'].value_counts())

# ==============================
# 4. Preprocessing
# ==============================
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

# Apply preprocessing on existing clean_text column
df['clean_text'] = df['clean_text'].apply(preprocess)

print("\nSample Cleaned Text:")
print(df[['clean_text']].head())

# ==============================
# 5. Train-Test Split
# ==============================
X = df['clean_text']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 6. Feature Engineering
# ==============================

# Bag of Words
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ==============================
# 7. Model Training + Evaluation
# ==============================

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

results = []

def evaluate_model(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    results.append([name, acc, prec, rec, f1])

# BoW Models
for name, model in models.items():
    evaluate_model(name + " (BoW)", model, X_train_bow, X_test_bow)

# TF-IDF Models
for name, model in models.items():
    evaluate_model(name + " (TF-IDF)", model, X_train_tfidf, X_test_tfidf)

# ==============================
# 8. Results Comparison
# ==============================
results_df = pd.DataFrame(
    results, 
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"]
)

print("\nModel Comparison:")
print(results_df)

# ==============================
# 9. Insights (Edit Based on Output)
# ==============================
print("""
Insights:
- TF-IDF generally performs better than Bag of Words.
- Logistic Regression usually gives best performance.
- Naive Bayes is fast but slightly less accurate.
- Decision Tree may overfit on text data.
""")

# ==============================
# 10. Save Results
# ==============================
results_df.to_csv('model_results.csv', index=False)

print("\nPipeline Completed Successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pavan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset Shape: (162973, 2)
                                          clean_text  category
0  when modi promised “minimum government maximum...      -1.0
1  talk all the nonsense and continue all the dra...       0.0
2  what did just say vote for modi  welcome bjp t...       1.0
3  asking his supporters prefix chowkidar their n...       1.0
4  answer who among these the most powerful world...       1.0

Columns: Index(['clean_text', 'category'], dtype='object')

Class Distribution:
category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64
